In [ ]:
import os
import pandas as pd
from typing import TypedDict, List, Dict, Any

from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings ChatOpenAI(
    model="openai/gpt-4o",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0.0,
)

embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-large",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    check_embedding_ctx_length=False,
)

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0.00011, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.00011, 'upstream_inference_prompt_cost': 2e-05, 'upstream_inference_completions_cost': 9e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o', 'system_fingerprint': 'fp_6d4f7d9eea', 'id': 'gen-1793246759-3Kd5ZCj7oF0OluWEijHc', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a185d6-09ac-7640-96ae-79545f4ad9d0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 

In [4]:
# Raw case documents mock
raw_documents = [
    Document(
        page_content="Entity: ACME Global Logistics LLC. KYC Status: Pending Verification - Missing Beneficial Owner ID. Date: Jan 10, 2026.",
        metadata={"case_id": "AML-2026-8891", "doc_type": "KYC", "page": 3, "file_name": "KYC_Report_ACME.pdf"}
    ),
    Document(
        page_content="Wire Transfer Log: On Jan 12, 2026, ACME Global Logistics transferred $450,000 USD to ShellCo Ltd (Cayman Islands).",
        metadata={"case_id": "AML-2026-8891", "doc_type": "WIRE_LOG", "page": 1, "file_name": "Wire_Transfers_Jan2026.csv"}
    ),
    Document(
        page_content="SAR Narrative: Suspicious rapid movement of funds observed within 48 hours of initial deposit by ACME Global Logistics LLC.",
        metadata={"case_id": "AML-2026-8891", "doc_type": "SAR", "page": 1, "file_name": "SAR_Narrative_8891.pdf"}
    )
]

# Ingest into FAISS Vector Store
vector_store = FAISS.from_documents(raw_documents, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [6]:
# 1. Define Agent State
class AuditState(TypedDict):
    case_id: str
    query: str
    retrieved_docs: List[Document]
    generated_analysis: str
    faithfulness_score: float
    retry_count: int

# 2. Define Retrieval Node
def retrieve_node(state: AuditState) -> Dict[str, Any]:
    query = state["query"]
    # Filter documents relevant to the current case ID
    docs = retriever.invoke(f"Case ID: {state['case_id']} - {query}")
    return {"retrieved_docs": docs}

# 3. Define Analysis & Citation Node
def analyze_node(state: AuditState) -> Dict[str, Any]:
    docs = state["retrieved_docs"]
    query = state["query"]
    
    context_str = "\n\n".join([
        f"Page: {d.metadata['page']}]\n{d.page_content}"
        for d in docs
    ])
    
    prompt = ChatPromptTemplate.from_template("""
    You are an expert AML/Fraud Internal Audit Assistant. 
    Analyze the provided case files and answer the question with explicit citations.
    
    Context Documents:
    {context}
    
    Audit Question: {question}
    
    Provide a concise analysis with source inline citations:
    """)
    
    chain = prompt | llm
    response = chain.invoke({"context": context_str, "question": query})
    return {"generated_analysis": response.content}

# 4. Evaluation Node using RAGAS Logic (Faithfulness check)
def evaluate_node(state: AuditState) -> Dict[str, Any]:
    # Simplified programmatic proxy for RAGAS Faithfulness evaluation
    analysis = state["generated_analysis"]
    docs = state["retrieved_docs"]
    
    # Check if key facts from docs exist in generation without hallucination
    ground_truth_claims = [d.page_content for d in docs]
    
    # Basic scoring logic: Ensure source filenames are explicitly cited
    citations_present = sum(1 for d in docs if d.metadata['file_name'] in analysis)
    score = citations_present / len(docs) if docs else 0.0
    
    return {"faithfulness_score": score, "retry_count": state.get("retry_count", 0) + 1}

# 5. Routing Condition
def decide_next_step(state: AuditState) -> str:
    if state["faithfulness_score"] >= 0.66 or state["retry_count"] >= 2:
        return "approved"
    return "retry"

# Build Graph
builder = StateGraph(AuditState)

builder.add_node("retrieve", retrieve_node)
builder.add_node("analyze", analyze_node)
builder.add_node("evaluate", evaluate_node)

builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "analyze")
builder.add_edge("analyze", "evaluate")

builder.add_conditional_edges(
    "evaluate",
    decide_next_step,
    {
        "approved": END,
        "retry": "retrieve"
    }
)

audit_agent = builder.compile()

In [7]:
# Run the pipeline for case AML-2026-8891
inputs = {
    "case_id": "AML-2026-8891",
    "query": "Did ACME Global Logistics engage in rapid layer-through transactions involving high-risk jurisdiction accounts after being flagged for incomplete KYC?",
    "retry_count": 0
}

output = audit_agent.invoke(inputs)

print("=== FINAL AUDIT SUMMARY ===")
print(output["generated_analysis"])
print("\n=== FAITHFULNESS METRIC ===")
print(f"RAGAS Verification Score: {output['faithfulness_score'] * 100:.1f}%")

=== FINAL AUDIT SUMMARY ===
Based on the provided case files, it appears that ACME Global Logistics LLC engaged in rapid layer-through transactions involving high-risk jurisdiction accounts after being flagged for incomplete KYC. The following points support this conclusion:

1. **Incomplete KYC Status**: ACME Global Logistics LLC's KYC status was pending verification due to a missing Beneficial Owner ID as of January 10, 2026 (Page 3). This indicates that the company was flagged for incomplete KYC.

2. **Suspicious Activity Report (SAR)**: A SAR narrative notes the suspicious rapid movement of funds within 48 hours of the initial deposit by ACME Global Logistics LLC (Page 1). This suggests that the company was involved in transactions that could be considered layering, a common money laundering technique.

3. **High-Risk Jurisdiction**: On January 12, 2026, ACME Global Logistics transferred $450,000 USD to ShellCo Ltd, which is located in the Cayman Islands (Page 1). The Cayman Island